# 05 · Train the NPE (GPU session, 2× T4)

Stops cleanly after `training.max_hours` (11 h). To continue, attach this notebook's previous output and rerun: the checkpoint is copied back and training resumes.

Record the date of the first *counted* run in `logs/research_log.md`.

In [ ]:
# --- Setup: get the code (no project logic in notebooks; everything lives in src/ and scripts/) ---
import os
import subprocess
import sys

REPO_URL = "https://github.com/powershell1/Almanac.git"
BRANCH = "master"
SRC = "/tmp/Almanac"
if os.path.exists("/kaggle/input/almanac-src/pyproject.toml"):
    SRC = "/kaggle/input/almanac-src"          # offline option: repo uploaded as a Dataset
elif not os.path.exists(SRC):
    url = REPO_URL
    try:                                       # private repo: add a Kaggle Secret GITHUB_TOKEN
        from kaggle_secrets import UserSecretsClient
        tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        url = REPO_URL.replace("https://", f"https://{tok}@")
    except Exception:
        pass
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, SRC], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "zuko==1.6.0"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", SRC], check=True)
os.chdir(SRC)
print(subprocess.run(["git", "log", "-1", "--format=%H %s"], capture_output=True, text=True).stdout)
WORK = "/kaggle/working"
CFG = "configs/killtest.yaml"


In [ ]:
SHARDS = "/kaggle/input/gdr4-03-simulate/shards/train"
PREV = "/kaggle/input/gdr4-05-train"          # optional resume
for sub in ("checkpoints", "results"):
    if os.path.exists(f"{PREV}/{sub}"):
        subprocess.run(["cp", "-rn", f"{PREV}/{sub}", WORK], check=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!torchrun --nproc_per_node 2 scripts/train.py --config {CFG} --root {WORK} --set paths.shards_dir={SHARDS}

In [ ]:
!tail -5 /kaggle/working/results/killtest_v1/train_log.csv